<a href="https://colab.research.google.com/github/MParvan/ecg-biometrics-bench/blob/main/experiments/Experiment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# First experiment: closed-set identification on ECG-ID

This notebook runs one complete benchmark end to end, so you can see
what the framework produces before reading any of its code.

The task is **closed-set identification**: the model is trained and
evaluated on the same set of subjects, and must assign each probe
heartbeat to one of them. It is the easiest of the eight protocols,
and deliberately so, because the interesting result is how far
performance falls when the protocol becomes realistic.

## 1. Setup

ECG-ID is downloaded automatically from PhysioNet on first use, so
there is nothing to fetch by hand.

Installation takes a few minutes, most of it PyTorch. On Colab pip
will report dependency conflicts against preinstalled packages such
as `opencv`, `jax`, and `shap`, which expect NumPy 2. **These
warnings are expected and harmless here.** The framework pins
NumPy 1.26 because that is the version its results were produced
with, and it does not use any of the conflicting packages.

In [ ]:
# Clone the framework and install its dependencies.
# On Colab this takes two to three minutes, mostly PyTorch.
!git clone https://github.com/MParvan/ecg-biometrics-bench.git
%cd ecg-biometrics-bench
!pip install -q -r requirements.txt

## 2. Run the experiment

`--epochs 10` keeps this notebook to a few minutes. The published
results use 250 epochs; see `configs/paper_reproduction/` for the
exact configurations.

In [ ]:
!python main.py \
  --dataset ecgid \
  --task 1 \
  --data_split_mode all-available \
  --model deepecg \
  --epochs 10 \
  --batch_size 256 \
  --n_runs 1 \
  --save_results

Expect Rank-1 accuracy around 0.70-0.80 from this short run.
The training loss is still falling at epoch 10, so the model is
deliberately undertrained: the point here is that the pipeline
runs, not that the number is competitive. Training to
convergence on this protocol reaches roughly 0.99.

## 3. What was produced

Every run writes a human-readable log and a machine-readable JSONL
record containing the metrics, the per-seed values, the resolved
configuration, the runtime profile, and the software environment.

In [ ]:
import json
from pathlib import Path

records = sorted(Path('../ecg-biometrics-artifacts/results')
                 .rglob('*.jsonl'))
print('result files:', [str(p) for p in records])

if records:
    latest = [json.loads(line)
              for line in records[-1].read_text(encoding='utf-8')
                                     .splitlines() if line.strip()][-1]
    print('\ntask   :', latest['task'])
    print('dataset:', latest['dataset'])
    print('\nresults:')
    for name, value in latest['results'].items():
        print(f'  {name:<22} {value}')

## Next

The number above is the optimistic case. To see what changes under a
realistic protocol, try `--task 3` (subject-disjoint: evaluated on
subjects never seen in training) or `--task 5` with a cross-session
split. `run_Module.ipynb` walks through all eight.